[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ame5003-nlp/blob/main/labs/lab-08-word-embeddings.ipynb)

**Click the badge above to open this lab in Google Colab.** Then choose *File → Save a copy in Drive* so your work is saved.

# Lab 8 — Word2Vec, GloVe, and the comparison

**MSIS · AME 5053 · Week 8 · 3 hours**

Lab 7 closed with two numbers and an open question: raw counts and Naive Bayes scored
**0.809**, sublinear TF-IDF scored **0.823**, and sessions 21–22 built two more ways to turn
a word into numbers without saying whether either one beats that. Session 21 went further
and gave a reason to worry: `cos(good, bad) = 0.719` in real Word2Vec vectors, so averaging a
review's word vectors together might cancel the very signal a sentiment classifier needs.

This lab does not defer the question again. It trains a real Word2Vec model, downloads real
GloVe vectors, and **measures** whether either beats TF-IDF on the same 2,000-review task,
with the same paired-trial protocol lab 7 used to tell a real result from noise. Whatever the
answer turns out to be, that is the answer this lab reports.

**By the end of this lab you will be able to:**

1. Train a Word2Vec model on your own corpus with `gensim`, and inspect what it learned
2. Load pretrained GloVe vectors and compare them to vectors you trained yourself
3. Turn a document into a single vector by averaging its word vectors
4. Run a fair, paired comparison between TF-IDF, Word2Vec and GloVe — and trust the result
5. Visualize word embeddings in two dimensions with PCA

---

## Part 0 — Setup

Same pattern as labs 6 and 7: NLTK's corpus download is a fresh network fetch every session.

In [ ]:
!pip install -q nltk scikit-learn gensim matplotlib

import nltk
ok = nltk.download("movie_reviews")
print("movie_reviews downloaded:", ok)

import numpy as np
import sklearn
import gensim
print("numpy", np.__version__, "· scikit-learn", sklearn.__version__, "· gensim", gensim.__version__)
print("Done.")

> **Save your own copy now:** File → Save a copy in Drive.

Same split as labs 6 and 7 — 1,500 documents to train on, 500 held out — so results in this
lab line up with what came before.

In [ ]:
from nltk.corpus import movie_reviews
from sklearn.model_selection import train_test_split

ids = movie_reviews.fileids()
texts = [movie_reviews.raw(i) for i in ids]
labels = np.array([1 if i.startswith("pos") else 0 for i in ids])

Xtr_text, Xte_text, ytr, yte = train_test_split(
    texts, labels, test_size=500, random_state=42, stratify=labels)

print("train:", len(Xtr_text), "  test:", len(Xte_text))

---

## Part 1 — Train your own Word2Vec (L2.5)

Session 21's vectors came from somebody else's cluster, trained on 100 billion words over
days. This part trains real ones **yourself**, on the only corpus this lab has: 1,500 movie
reviews. Session 21's arithmetic said a billion-token pass needs negative sampling to be
tractable — `gensim.models.Word2Vec` uses negative sampling by default, and a corpus this
small trains in a few seconds regardless.

First, a tokenizer. Reuse the simple lowercase-letters-only pattern rather than NLTK's
tokenizer — Word2Vec does not care about punctuation tokens, and this keeps the vocabulary
smaller and training faster.

In [ ]:
import re

token_re = re.compile(r"[a-z]+")

def tokenize(text):
    """Lowercase, keep runs of letters only."""
    return token_re.findall(text.lower())

tokenized_train = [tokenize(t) for t in Xtr_text]
n_tokens = sum(len(t) for t in tokenized_train)
print("training documents:", len(tokenized_train))
print("training tokens:", n_tokens)
print("first document, first 12 tokens:", tokenized_train[0][:12])

Now train it. `gensim.models.Word2Vec` takes a list of tokenized documents directly — no
manual window-sliding, no manual negative sampling, all of session 21's machinery is inside
this one call. The hyperparameters to use:

```
vector_size = 100     # smaller than session 21's 300 — this corpus is much smaller too
window      = 5
min_count   = 5        # drop words appearing fewer than 5 times — mostly noise on 1,500 docs
sg          = 1        # skip-gram: session 21 said "if you are choosing blind, choose skip-gram"
epochs      = 10
seed        = 42
```

Write the function that trains it.

In [ ]:
from gensim.models import Word2Vec

def train_word2vec(tokenized_docs):
    """Train a Word2Vec model on a list of tokenized documents. Returns the trained model."""
    # YOUR CODE HERE
    # Word2Vec(sentences=..., vector_size=100, window=5, min_count=5, sg=1, epochs=10, seed=42)
    pass

### Check the neighbours

Trained on 1,500 reviews instead of 100 billion words, so do not expect session 21's polish
— but the same idea should already be visible. Print the five nearest neighbours (by cosine)
of `terrible`, `great`, `movie` and `actor`.

In [ ]:
def print_neighbours(kv, words, topn=5):
    for w in words:
        if w in kv:
            neighbours = ", ".join(f"{n} {c:.3f}" for n, c in kv.most_similar(w, topn=topn))
            print(f"{w:10s} -> {neighbours}")
        else:
            print(f"{w:10s} -> not in vocabulary")

# YOUR CODE HERE
# call print_neighbours(w2v.wv, ["terrible", "great", "movie", "actor"])

Noisier than session 21's GoogleNews vectors, and it should be — 1,500 documents is nothing
next to 100 billion words. But the structure is already real: `terrible` pulls in
`atrocious`, `crappy` and `amateurish`; `movie` pulls in `film`. **1,500 reviews were enough
to learn the shape of the distributional hypothesis, just not enough to learn it cleanly.**
`rochon` and `malone` next to `actor` are almost certainly actors' surnames the corpus
happens to mention often — a reminder that small-corpus Word2Vec learns whatever the corpus
contains, including its idiosyncrasies.

---

## Part 2 — Download real GloVe vectors (L2.6)

Session 22 built these from a 6-billion-token corpus — a thousand times more text than Part
1's entire training set. `gensim.downloader` fetches the same file session 22 used
(`glove-wiki-gigaword-100` — the 100-dimensional release, smaller and faster to download than
session 22's 300-dimensional one; still Wikipedia + Gigaword, still 400,000 words).

In [ ]:
import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-100")   # ~128MB — the slow cell, run it once
print("GloVe vocabulary size:", len(glove))

Same four words, same function, different vectors.

In [ ]:
# YOUR CODE HERE
# call print_neighbours(glove, ["terrible", "great", "movie", "actor"])

Cleaner than Part 1's, as 1,000× more training data should make it — `terrible`'s
neighbours are tightly synonymous (`horrible`, `awful`, `dreadful`) rather than loosely
related. But look at `great`: `good`, `little`, `well` — plain frequent words with no
particular sentiment content of their own, the same pattern session 22 found for `good`
itself (`really`, `you`, `well`, `things` — none of them synonyms, all of them just common).
**Bigger training data does not fix every problem** — it fixes sparsity, not the fact that
very frequent words end up near lots of other frequent words regardless of what they mean.

---

## Part 3 — From word vectors to document vectors

Both models give a vector **per word**. A classifier needs a vector **per review**. The
standard move — the one this lab tests — is to **average** the word vectors of every word in
the document.

Session 21 already flagged the risk in doing this for sentiment: `cos(good, bad) = 0.719` in
real Word2Vec vectors. Average a positive review's words with a negative review's words and
whatever made them different partially cancels. TF-IDF never does this — every word keeps its
own separate feature, however the words relate to each other. Part 4 measures whether that
difference matters.

In [ ]:
def doc_vector(tokens, kv):
    """Average the vectors of tokens found in kv. Returns a zero vector if none are found."""
    # YOUR CODE HERE
    # look up kv[t] for each token t that is "in kv", collect them, and average.
    # If nothing was found, return np.zeros(kv.vector_size) — never divide by zero.
    pass

In [ ]:
tokenized_test = [tokenize(t) for t in Xte_text]

Xtr_w2v = np.array([doc_vector(t, w2v.wv) for t in tokenized_train])
Xte_w2v = np.array([doc_vector(t, w2v.wv) for t in tokenized_test])
Xtr_glove = np.array([doc_vector(t, glove) for t in tokenized_train])
Xte_glove = np.array([doc_vector(t, glove) for t in tokenized_test])

print("Word2Vec document matrix:", Xtr_w2v.shape)
print("GloVe document matrix:   ", Xtr_glove.shape)

---

## Part 4 — The comparison (L2.7 — the question this lab exists to answer)

Two representations built. One already exists: TF-IDF, from lab 7. Time to find out which
wins.

**One change from lab 7's protocol.** `MultinomialNB` — what lab 6 and lab 7 both used —
requires non-negative input, and averaged word vectors have negative entries. Rather than
inventing a different classifier for each representation (which would make the comparison
meaningless — a difference in scores could just be a difference in classifiers), this lab
uses **`LogisticRegression` for all three**, TF-IDF included. So the numbers below will not
match lab 6/7's NB numbers exactly, and that is expected, not an error.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf = TfidfVectorizer(sublinear_tf=True)
Xtr_tfidf = tfidf.fit_transform(Xtr_text)
Xte_tfidf = tfidf.transform(Xte_text)

def accuracy_lr(Xtr, Xte):
    clf = LogisticRegression(max_iter=2000)
    clf.fit(Xtr, ytr)
    return clf.score(Xte, yte)

print("single 42-split accuracy, LogisticRegression throughout:")
print("  TF-IDF (sublinear)          :", round(accuracy_lr(Xtr_tfidf, Xte_tfidf), 4))
print("  Word2Vec (on-corpus, avg)   :", round(accuracy_lr(Xtr_w2v, Xte_w2v), 4))
print("  GloVe (pretrained, avg)     :", round(accuracy_lr(Xtr_glove, Xte_glove), 4))

# Verified output:
#   TF-IDF (sublinear)          : 0.844
#   Word2Vec (on-corpus, avg)   : 0.732
#   GloVe (pretrained, avg)     : 0.722
# For context only (different classifier, not part of the paired comparison below):
#   raw counts + Naive Bayes (lab 6 style)      : 0.824
#   sublinear TF-IDF + Naive Bayes (lab 7 style): 0.824

One split is not enough — lab 7's Part 6 spent an hour establishing that. Run the same
paired-trial protocol: **20 seeds, same train/test split shared across all three
representations at each seed**, Word2Vec retrained fresh every time (it depends on the
training split), GloVe reused as-is (it does not depend on the split at all — it was already
trained before this lab began).

In [ ]:
def paired_trial_three(n=20):
    """Score TF-IDF, Word2Vec and GloVe on the same n splits. Returns three lists of accuracies."""
    # YOUR CODE HERE
    # For each seed in range(n):
    #   1. train_test_split(texts, labels, test_size=500, random_state=seed, stratify=labels)
    #   2. fit a fresh TfidfVectorizer(sublinear_tf=True) on this split's training texts
    #   3. tokenize this split's texts, train_word2vec on the training tokens
    #   4. build doc_vector matrices for w2v (freshly trained) and glove (already loaded)
    #   5. fit LogisticRegression on each of the three training matrices, score on the test one
    #   6. append each accuracy to its list
    pass

In [ ]:
def wins_losses_ties(a, b):
    w = sum(1 for x, y in zip(a, b) if x > y)
    l = sum(1 for x, y in zip(a, b) if x < y)
    t = sum(1 for x, y in zip(a, b) if x == y)
    return w, l, t

print("TF-IDF beats Word2Vec on:", wins_losses_ties(tfidf_acc, w2v_acc), "(win/loss/tie splits)")
print("TF-IDF beats GloVe on:   ", wins_losses_ties(tfidf_acc, glove_acc))
print("GloVe beats Word2Vec on: ", wins_losses_ties(glove_acc, w2v_acc))

# Verified output:
#   TF-IDF beats Word2Vec on: (20, 0, 0)
#   TF-IDF beats GloVe on:    (20, 0, 0)
#   GloVe beats Word2Vec on:  (7, 12, 1)

### Read this the way lab 7 taught you to

**TF-IDF wins, and it is not close.** 20 splits out of 20 against Word2Vec, 20 out of 20
against GloVe. The gap — about **0.12** — is roughly **seven times** the split-to-split noise
(std ≈ 0.016–0.021). This is exactly the shape of result lab 7 called "not weak": a record
this one-sided, with an effect this much larger than the noise, is a real finding, not a
coin-flip that happened to land one way.

**Sessions 20 and 21 both flagged this and declined to answer it. Here is the answer: no,
averaged embeddings do not beat TF-IDF on this task.** The reason is the one session 21 gave
before any of this ran: averaging a review's word vectors blends its positive and negative
words into one point, and `cos(good, bad) = 0.719` means that blending has real room to erase
the signal a sentiment classifier needs. TF-IDF keeps `good` and `bad` as two separate
features and never merges them.

**GloVe vs. Word2Vec is the opposite shape of result, and reporting it as decisive would be
the mistake lab 7 warned against.** On-corpus Word2Vec — trained on 1,500 documents, no
external data at all — edges out pretrained GloVe on 12 of 20 splits. But the effect
(≈0.007) is *smaller* than the noise (≈0.015–0.021), so this is a tendency worth mentioning,
not a result worth claiming. If it holds up, a plausible reason is that Word2Vec here learned
vocabulary specific to *this* domain (`atrocious`, `crappy`, `amateurish` clustering with
`terrible`) that a generic Wikipedia + Gigaword corpus had less reason to capture as sharply
— but 12–7–1 is not strong enough evidence to hang that story on with confidence.

---

## Part 5 — Visualize the vectors (L2.8)

300 or 100 numbers per word cannot be looked at directly. **PCA** finds the 2 directions
that capture the most variation across a set of vectors and projects onto just those —
throwing most of the information away, on purpose, in exchange for something you can plot.

Pick a handful of words from a few different groups and see whether the plot recovers the
groups without being told about them.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

words = [
    "good", "great", "excellent", "wonderful",     # positive
    "bad", "terrible", "awful", "horrible",        # negative
    "king", "queen", "prince", "princess",         # royalty
    "car", "truck", "bicycle", "train",            # vehicles
]

# YOUR CODE HERE
# 1. Build a matrix of GloVe vectors, one row per word in `words` (all are in vocab).
# 2. PCA(n_components=2).fit_transform(...) that matrix.
# 3. Scatter the 2D points and label each with its word (plt.annotate).

**Expect the four groups to cluster** — vehicles near each other, royalty near each other,
positive words near each other, negative words near each other — because that clustering is
exactly what cosine similarity has been measuring all lab. **Also expect `good` and `bad`
closer together than feels right**, for the reason this whole lab has been about: they share
company, and a method built on company cannot fully pull them apart.

**A note on the two other tools L2.8 mentions.** **t-SNE**
(`sklearn.manifold.TSNE`) is nonlinear and often separates clusters more crisply than PCA —
but it is stochastic (a different `random_state` gives a visibly different layout) and its
*distances* are not meaningful the way PCA's are, only the *clusters* are. Try swapping
`PCA(n_components=2)` for `TSNE(n_components=2, random_state=0, perplexity=5)` above and
compare. **The [TensorFlow Embedding Projector](https://projector.tensorflow.org/)** is a
browser tool for exploring pretrained embeddings interactively in 3D — worth five minutes
outside the lab, not required to finish it.

---

## Where this leaves Unit II

Three representations of a document, one task, one honest measurement:

| representation | classifier | mean accuracy (20 splits) |
|---|---|---|
| raw counts (lab 6) | Naive Bayes | 0.809 |
| sublinear TF-IDF (lab 7) | Naive Bayes | 0.823 |
| sublinear TF-IDF (this lab) | Logistic Regression | 0.851 |
| Word2Vec, on-corpus, averaged | Logistic Regression | 0.731 |
| GloVe, pretrained, averaged | Logistic Regression | 0.724 |

However you cut it, **TF-IDF wins**, by a wide and well-supported margin. Dense, learned
vectors did not beat sparse, counted ones on this task — not because Word2Vec and GloVe are
bad representations of *words* (Parts 1 and 2 showed they clearly are not), but because
**averaging destroys exactly the information a sentiment task needs**. That is the honest
answer sessions 20 and 21 both deferred, and it is a genuinely useful thing to know before
reaching for embeddings on the next sentiment problem: averaging is not free.

It is also not the end of the story. Unit III opens with representations that do not throw
sentence order away the way averaging does — the first step toward embeddings that know
which word comes next, not just which words keep company.